Optimization Goal: Maximise

Description of Sample Application:

You’re working on a drug discovery project, testing combinations of three compounds to create a new medicine.

Each experiment is stored as a 3D array, where each row lists the amounts of the three compounds used. After each experiment, you record the number of adverse reactions, stored in as a 1D array.

Your goal is to minimise side effects; in this competition, it is framed as maximisation by optimising a transformed output (e.g. the negative of side effects).

In [1]:
# =============================================================================
# IMPORTS AND SETTINGS
# =============================================================================

import warnings

import numpy as np
import pandas as pd

from scipy.spatial.distance import cdist
from scipy.stats import norm, qmc, rankdata

from sklearn.exceptions import ConvergenceWarning
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel as C, RBF, Matern, WhiteKernel
from sklearn.preprocessing import PowerTransformer

warnings.filterwarnings("ignore", category=ConvergenceWarning)

RANDOM_STATE = 42
FUNCTION_ID = 3

# Surrogates: ARD GPs on a Yeo-Johnson transformed target, one per kernel.
# (Grouped leave-one-out CV preferred Yeo-Johnson over the raw target for both.)
KERNELS = ["RBF", "Matern_3_2"]
NOISE_BOUNDS = (1e-6, 1e0)     # WhiteKernel bounds
N_RESTARTS = 10                # optimiser restarts for each final GP fit

# Acquisition functions, in ORIGINAL y units
XI = 0.0                       # EI / PI margin (y <= 0 here, so keep near 0)
BETA = 2.0                     # UCB = upper BETA-sigma quantile of y
N_DRAWS = 512                  # posterior quantile draws per point used for EI / PI / Std
SOBOL_POWER = 18               # 2^18 candidates per population

# Search geometry
TOP_K_LOCAL = 5                # local region = box around the K best observations (padded by data spacing)
CORRIDOR_RADIUS_FRAC = 0.25    # corridor tube radius as a fraction of median nearest-neighbour spacing
TOP_N = 10                     # rows shown in candidate tables


def fmt(df):
    return df.to_string(index=False, float_format=lambda v: f"{v:.6f}")


def format_point(x):
    """Format a point as the dash-separated six-decimal submission string."""
    return "-".join(f"{v:.6f}" for v in x)

In [2]:
# =============================================================================
# DATA LOADING
# =============================================================================
# Loads the centrally prepared BBO dataset and selects the active function.
# Active input dimensions are inferred from the x-columns that contain data.

DATA_PATH = "../data/data.csv"

data = pd.read_csv(DATA_PATH, float_precision="round_trip")
data = data[data["function"] == FUNCTION_ID].copy()

feature_cols = [f"x{i}" for i in range(1, 9) if data[f"x{i}"].notna().any()]
X = data[feature_cols].to_numpy()
y = data["y"].to_numpy()
rounds = data["round"].to_numpy()
n_dim = X.shape[1]

incumbent_idx = int(np.argmax(y))
incumbent_x = X[incumbent_idx].copy()
incumbent_y = float(y[incumbent_idx])

# =============================================================================
# DATA-DRIVEN SEARCH REGIONS
# =============================================================================
#   spacing      = median nearest-neighbour distance between distinct observed points
#   LOCAL_BOUNDS = box around the TOP_K_LOCAL best observations, padded by one spacing

nn = cdist(X, X)
nn[nn == 0] = np.inf
spacing = float(np.median(nn.min(axis=1)))
top_k = np.argsort(y)[-TOP_K_LOCAL:]
local_lo = np.clip(X[top_k].min(axis=0) - spacing, 0.0, 1.0)
local_hi = np.clip(X[top_k].max(axis=0) + spacing, 0.0, 1.0)
LOCAL_BOUNDS = list(zip(local_lo, local_hi))

print(f"Observations: {len(y)} | Features: {feature_cols}")
print(f"Incumbent: x={incumbent_x} | y={incumbent_y:.9f} | round={rounds[incumbent_idx]}")
print(f"Median spacing: {spacing:.4f}")
print("Local bounds: " + ", ".join(f"{f} [{lo:.3f}, {hi:.3f}]" for f, (lo, hi) in zip(feature_cols, LOCAL_BOUNDS)))

Observations: 26 | Features: ['x1', 'x2', 'x3']
Incumbent: x=[0.628331 0.443178 0.440136] | y=-0.007172234 | round=9
Median spacing: 0.1521
Local bounds: x1 [0.262, 0.790], x2 [0.262, 0.652], x3 [0.248, 0.669]


In [3]:
# =============================================================================
# SURROGATE MODELS AND ACQUISITION FUNCTIONS
# =============================================================================


def make_kernel(name, n_dims):
    """Constant * ARD kernel + white noise."""
    if name == "RBF":
        base = RBF(np.ones(n_dims), length_scale_bounds=(1e-2, 1e4))
    elif name == "Matern_3_2":
        base = Matern(np.ones(n_dims), length_scale_bounds=(1e-2, 1e4), nu=1.5)
    return C(1.0, (1e-3, 1e3)) * base + WhiteKernel(1e-2, NOISE_BOUNDS)


def fit_surrogate(X_train, y_train, kernel_name):
    """Fit a GP on the Yeo-Johnson transformed target. Returns (gp, transformer)."""
    pt = PowerTransformer(method="yeo-johnson", standardize=False)
    y_t = pt.fit_transform(np.asarray(y_train).reshape(-1, 1)).ravel()
    gp = GaussianProcessRegressor(
        kernel=make_kernel(kernel_name, X_train.shape[1]),
        alpha=1e-10,                    # numerical jitter only
        normalize_y=True,
        n_restarts_optimizer=N_RESTARTS,
        random_state=RANDOM_STATE,
    ).fit(X_train, y_t)
    return gp, pt


def from_t(pt, values):
    """Array on the transformed scale -> original y."""
    return pt.inverse_transform(np.asarray(values).reshape(-1, 1)).ravel()


def predict_batch(model_bundle, X_query, batch_size=65536):
    """Transformed-scale posterior mean/std, batched to bound memory."""
    gp, _ = model_bundle
    out = [gp.predict(X_query[i:i + batch_size], return_std=True) for i in range(0, len(X_query), batch_size)]
    return np.concatenate([m for m, _ in out]), np.concatenate([s for _, s in out])


# EI and PI are not preserved by the nonlinear transform, so they are evaluated on the
# original y scale: average over N_DRAWS evenly spaced posterior quantiles mapped back to y.
#   Pred_y = median posterior (exact, the transform is monotone)
#   UCB    = upper BETA-sigma quantile of y (exact, quantiles survive a monotone transform)
#   Std    = std of the y draws (Std_t is the transformed-scale std)
# Draws outside the transform's valid range are impossible y values far below the
# incumbent, so they count as zero improvement.
Z_DRAWS = norm.ppf((np.arange(N_DRAWS) + 0.5) / N_DRAWS)


def acquisition_y(model_bundle, X_query, xi=XI, beta=BETA):
    """Posterior summaries and acquisition values, all in original y units."""
    _, pt = model_bundle
    mu_t, std_t = predict_batch(model_bundle, X_query)
    n = len(mu_t)
    ei, pi, sd_y = np.empty(n), np.empty(n), np.empty(n)
    step = max(1, 2_000_000 // N_DRAWS)               # bound memory: points x draws per chunk
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")               # sklearn warns about NaN inverses (handled below)
        for i in range(0, n, step):
            s = slice(i, i + step)
            draws = mu_t[s, None] + std_t[s, None] * Z_DRAWS[None, :]
            y_d = pt.inverse_transform(draws.reshape(-1, 1)).reshape(draws.shape)
            gain = y_d - incumbent_y - xi
            ei[s] = np.nan_to_num(np.maximum(gain, 0.0), nan=0.0).mean(axis=1)
            pi[s] = (gain > 0).mean(axis=1)           # NaN > 0 is False
            sd_y[s] = np.nanstd(y_d, axis=1)
        pred = from_t(pt, mu_t)
        ucb = from_t(pt, mu_t + beta * std_t)
    return {"Pred_y": pred, "Std": sd_y, "Std_t": std_t, "EI": ei, "PI": pi, "UCB": ucb}


# ---- Fit the working surrogates -------------------------------------------------
models = {name: fit_surrogate(X, y, name) for name in KERNELS}

rows = []
for name, (gp, pt) in models.items():
    fitted = gp.kernel_
    rows.append({
        "Model": name,
        "YJ_lambda": float(pt.lambdas_[0]),
        "Log_Marginal_Likelihood": float(gp.log_marginal_likelihood_value_),
        "Noise_Level": float(fitted.k2.noise_level),
        **{f"{f}_LengthScale": float(ls) for f, ls in zip(feature_cols, np.atleast_1d(fitted.k1.k2.length_scale))},
    })
print("WORKING SURROGATES")
print(fmt(pd.DataFrame(rows)))

WORKING SURROGATES
     Model  YJ_lambda  Log_Marginal_Likelihood  Noise_Level  x1_LengthScale  x2_LengthScale  x3_LengthScale
       RBF  12.917801               -21.295729     0.056703        0.649185        0.460634        0.116553
Matern_3_2  12.917801               -22.016694     0.055841        1.092141        0.839422        0.177502


In [4]:
# =============================================================================
# CANDIDATE SEARCH AND RECOMMENDED POINT
# =============================================================================
# Every candidate is scored by both surrogates under four criteria (posterior
# mean, EI, PI, UCB). Consensus = mean percentile rank across those 8 scores;
# Min_Support = the weakest of the 8 ranks. Distance to the nearest observation
# is reported but deliberately NOT part of the consensus.


def score_candidates(cands):
    """Score points with every surrogate: coordinates, per-model Pred_y/Std/EI/PI/UCB,
    Consensus, Min_Support and Nearest_Distance."""
    table = pd.DataFrame(cands, columns=feature_cols)
    table["Nearest_Distance"] = cdist(cands, X).min(axis=1)
    n = len(cands)
    ranks = []
    for name, bundle in models.items():
        cols = acquisition_y(bundle, cands)
        for k, v in cols.items():
            table[f"{name}_{k}"] = v
        ranks += [(rankdata(cols[k], method="ordinal") - 1) / (n - 1) for k in ("Pred_y", "EI", "PI", "UCB")]
    ranks = np.column_stack(ranks)
    table["Consensus"] = ranks.mean(axis=1)
    table["Min_Support"] = ranks.min(axis=1)
    return table


def individual_optima(table):
    """Best candidate for each (model, criterion) pair."""
    rows = []
    for name in models:
        for crit in ("Pred_y", "EI", "PI", "UCB"):
            r = table.loc[table[f"{name}_{crit}"].idxmax()]
            rows.append({"Model": name, "Criterion": crit, **r[feature_cols].to_dict(),
                         "Predicted_y": r[f"{name}_Pred_y"], "Std": r[f"{name}_Std"],
                         "EI": r[f"{name}_EI"], "PI": r[f"{name}_PI"], "UCB": r[f"{name}_UCB"],
                         "Nearest_Distance": r["Nearest_Distance"]})
    return pd.DataFrame(rows)


pred_cols = [f"{m}_Pred_y" for m in models]
ei_cols = [f"{m}_EI" for m in models]
summary_cols = [*feature_cols, "Consensus", "Min_Support", "Nearest_Distance", *pred_cols, *ei_cols]

# ---- Global search: scrambled Sobol sample of the full domain -------------------
cands = qmc.Sobol(d=n_dim, scramble=True, seed=RANDOM_STATE).random_base2(m=SOBOL_POWER)
pop = score_candidates(cands)

print(f"GLOBAL SEARCH | domain [0,1]^{n_dim} | {len(pop):,} candidates | {len(X)} observations")
print(f"Incumbent: {format_point(incumbent_x)} | y={incumbent_y:.9f}")
print("\nTOP CONSENSUS CANDIDATES")
print(fmt(pop.nlargest(TOP_N, "Consensus")[summary_cols]))
print("\nINDIVIDUAL MODEL / CRITERION OPTIMA")
print(fmt(individual_optima(pop)))

# ---- Corridor search between the incumbent and the next-best distinct region ----
point_A = incumbent_x
far = [i for i in np.argsort(y)[::-1] if np.linalg.norm(X[i] - point_A) > spacing]
point_B = X[far[0]] if far else X[np.argsort(y)[-2]]
vec = point_B - point_A
radius = CORRIDOR_RADIUS_FRAC * spacing

print("\n\nCORRIDOR SEARCH")
print(f"Point A (incumbent):         {np.round(point_A, 6)}")
print(f"Point B (best distinct obs): {np.round(point_B, 6)}")
print(f"Corridor length: {np.linalg.norm(vec):.6f} | tube radius: {radius:.4f}")

s = qmc.Sobol(d=1 + n_dim, scramble=True, seed=RANDOM_STATE).random_base2(SOBOL_POWER)
t_line = s[:, 0]                                                   # 0 = A, 1 = B
corr_pts = np.clip(point_A + t_line[:, None] * vec + (s[:, 1:] * 2 - 1) * radius, 0.0, 1.0)
corr = score_candidates(corr_pts)
corr["Corridor_t"] = ((corr_pts - point_A) @ vec) / (vec @ vec)
corr["Distance_from_line"] = np.linalg.norm(corr_pts - (point_A + corr["Corridor_t"].to_numpy()[:, None] * vec), axis=1)

print("\nBEST CORRIDOR CANDIDATES")
corr_best = individual_optima(corr)
print(fmt(corr_best[corr_best.Criterion.isin(["Pred_y", "EI", "UCB"])]))
print("\nTOP CONSENSUS CORRIDOR CANDIDATES")
print(fmt(corr.nlargest(10, "Consensus")[[*feature_cols, "Consensus", "Corridor_t", "Distance_from_line", *pred_cols]]))

# ---- Recommended query: the highest-consensus corridor candidate ----------------
x_next3 = corr.loc[corr["Consensus"].idxmax(), feature_cols].to_numpy(dtype=float)
print("\nRecommended next query (top consensus corridor candidate):")
print(format_point(x_next3))

GLOBAL SEARCH | domain [0,1]^3 | 262,144 candidates | 26 observations
Incumbent: 0.628331-0.443178-0.440136 | y=-0.007172234

TOP CONSENSUS CANDIDATES
      x1       x2       x3  Consensus  Min_Support  Nearest_Distance  RBF_Pred_y  Matern_3_2_Pred_y   RBF_EI  Matern_3_2_EI
0.514876 0.513231 0.427102   0.987564     0.945976          0.075568   -0.013830          -0.014176 0.000605       0.000723
0.493655 0.523488 0.428798   0.987418     0.946560          0.075244   -0.014162          -0.014697 0.000597       0.000711
0.857543 0.313985 0.492230   0.987271     0.969162          0.196472   -0.019918          -0.020909 0.000652       0.000739
0.539264 0.524436 0.425986   0.987249     0.944706          0.072277   -0.013853          -0.013992 0.000593       0.000732
0.879478 0.316193 0.491190   0.987198     0.971088          0.212728   -0.020382          -0.021281 0.000661       0.000753
0.518945 0.503380 0.423610   0.987177     0.944462          0.078440   -0.013792          -0.013946 0.000


BEST CORRIDOR CANDIDATES
     Model Criterion       x1       x2       x3  Predicted_y      Std       EI       PI       UCB  Nearest_Distance
       RBF    Pred_y 0.596587 0.459065 0.437787    -0.013049 0.006809 0.000613 0.183594 -0.000495          0.035575
       RBF        EI 0.569314 0.476675 0.435392    -0.013120 0.006908 0.000623 0.185547 -0.000401          0.041128
       RBF       UCB 0.538265 0.473776 0.435565    -0.013265 0.007008 0.000619 0.181641 -0.000377          0.063088
Matern_3_2    Pred_y 0.590333 0.478334 0.417347    -0.013360 0.007333 0.000676 0.189453  0.000071          0.017430
Matern_3_2        EI 0.561293 0.476049 0.417477    -0.013440 0.007478 0.000695 0.189453  0.000232          0.033839
Matern_3_2       UCB 0.544065 0.474228 0.421042    -0.013587 0.007592 0.000693 0.187500  0.000274          0.050854

TOP CONSENSUS CORRIDOR CANDIDATES
      x1       x2       x3  Consensus  Corridor_t  Distance_from_line  RBF_Pred_y  Matern_3_2_Pred_y
0.565436 0.475753 0.431480